# Simulacao_App — dados do app calibrados nos dados reais

O app Caminho dos Direitos ainda é um protótipo, então não existem casos de verdade. Este notebook **simula** o que os agentes registrariam no app, mas **não inventa as proporções**: cada probabilidade vem das bases públicas que o grupo já tratou (`saida/`) e do modelo (`saida_modelo/`).

**Ordem para rodar:** `Tratamento_GS_v5` → `Modelo_GS_v4` → este notebook.

| O que é simulado | Calibrado com (dado real) |
|---|---|
| Quantos casos cada distrito tem | Famílias no CadÚnico do distrito (SMADS), com pelo menos 1 caso em cada um dos 96 |
| Unidade (UBS/CRAS) do caso | Unidades do próprio distrito (`dim_equipamento`, CNES/GeoSampa) |
| Caso em área de risco R3/R4 | Moradias em setores R3/R4 ÷ domicílios do distrito (GeoSampa + Censo) |
| Idoso 65+ | % de 65+ do distrito (Censo 2022) |
| Já recebe Bolsa Família / BPC | Cobertura real ÷ cobertura esperada do distrito (modelo): onde o modelo vê lacuna, mais gente sai como "não recebe". É por programa: um distrito urgente por causa do Bolsa Família pode não ter lacuna de BPC |
| Falta documento | Distância do distrito ao cartório e ao Poupatempo (`fato_acesso_distrito_servico`) |
| Faltou ao encaminhamento | Distância do distrito ao serviço de destino |

O que **não** tem dado público por distrito (renda por pessoa, faixas etárias abaixo de 65, peso de cada documento, Tarifa Social, Pé-de-Meia, Farmácia Popular) usa as **premissas** da célula de configuração, todas num lugar só para o grupo revisar.

Toda linha sai com `origem = "SIMULADO"`. No painel, esses blocos levam o selo SIMULADO.

**Saídas** (`Dados/saida_app/`, CSV com `;` e vírgula decimal): `dim_caso`, `fato_item_plano`, `fato_documento_pendente`, `fato_encaminhamento`, `fato_alertas`.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = Path('/content/drive/MyDrive/Dados')
except ImportError:                      # rodando fora do Colab
    PASTA = Path(os.environ.get('PASTA_DADOS', 'Dados'))

PASTA_SAIDA_TRAT = PASTA / 'saida'
PASTA_MODELO = PASTA / 'saida_modelo'
PASTA_APP = PASTA / 'saida_app'
PASTA_APP.mkdir(parents=True, exist_ok=True)

SEMENTE = 42
rng = np.random.default_rng(SEMENTE)

Mounted at /content/drive


## Configuração e premissas
Tudo o que **não** vem de dado público está aqui. Se a pesquisa do Google Forms tiver números melhores (ex.: qual documento mais falta), troquem os valores.

In [2]:
N_CASOS = 5000                 # pessoas acompanhadas pelos agentes na simulação
PCT_CASOS_ENCERRADOS = 0.10    # parte dos casos já encerrados (o painel mostra só os ativos)

# Renda por pessoa (faixas do app). Premissa: perfil do CadÚnico de SP, deslocado para baixo nos distritos com mais CadÚnico por habitante
FAIXAS_RENDA = ['até R$ 218', 'até R$ 405', 'até R$ 810', 'até R$ 1.621', 'acima', 'não informado']
PROP_RENDA = np.array([0.42, 0.16, 0.20, 0.13, 0.05, 0.04])

# Faixa etária: a fatia 65+ vem do Censo por distrito; o resto se divide nestas proporções (premissa)
FAIXAS_IDADE = ['0–5', '6–17', '18–29', '30–64', '65+']
PROP_IDADE_SEM_65 = np.array([0.09, 0.20, 0.17, 0.54])
SOBRE_REPRESENTACAO_65 = 1.3   # agentes visitam mais idosos do que a média da população

# Documento: chance base de faltar pelo menos 1, que cresce com a distância ao cartório/Poupatempo
P_SEM_DOC_BASE = 0.12
P_SEM_DOC_POR_KM = 0.012       # +1,2 ponto por km de distância média (cartório + Poupatempo)
PROP_DOCUMENTO = {'RG/CIN': .30, 'Certidão de nascimento': .22, 'Comprovante de residência': .20,
                  'CPF': .16, 'Título de eleitor': .07, 'Carteira de trabalho': .05}

# Risco: moradias R3/R4 ÷ domicílios; famílias do CadÚnico moram mais em área de risco que a média
PESSOAS_POR_DOMICILIO = 2.8
SOBRE_REPRESENTACAO_RISCO = 2.0

# Programas sem dado por distrito: chance de já receber entre quem tem direito (premissa, nível cidade)
P_RECEBE_FIXO = {'CadÚnico': 0.80, 'Tarifa Social de Energia': 0.60, 'Pé-de-Meia': 0.55, 'Farmácia Popular': 0.50}

# Entre quem tem direito e não recebe, como o agente marca (premissa)
PROP_NAO_RECEBE = {'Ainda não pediu': 0.45, 'Precisa atualizar': 0.30, 'Bloqueado': 0.25}

# Encaminhamentos
P_ENCAMINHA = 0.70             # itens sem acesso que o agente encaminha
P_FALTA_BASE = 0.10
P_FALTA_POR_KM = 0.04          # +4 pontos de falta por km até o serviço
PROP_RESULTADO_SEM_FALTA = {'Concluído': 0.50, 'Compareceu': 0.25, 'Agendado': 0.25}
DESTINO = {'Documentação básica': None,  # cartório (certidão) ou Poupatempo (RG, CPF), conforme o documento
           'CadÚnico': 'CRAS', 'Bolsa Família': 'CRAS', 'BPC': 'INSS',
           'Tarifa Social de Energia': 'CRAS', 'Pé-de-Meia': 'CRAS', 'Farmácia Popular': 'UBS'}
DIST_DESTINO = {'CRAS': 'CRAS', 'UBS': 'UBS', 'Cartório': 'Cartório', 'Poupatempo': 'Poupatempo',
                'INSS': 'Poupatempo'}   # sem base de agências do INSS: usa a distância ao Poupatempo como aproximação

ORDEM_SITUACAO = {'Já recebe': 1, 'Precisa atualizar': 2, 'Ainda não pediu': 3, 'Falta documento': 4, 'Bloqueado': 5}
ORDEM_RESULTADO = {'Agendado': 1, 'Compareceu': 2, 'Concluído': 3, 'Faltou': 4}

## 1. Ler as bases reais

In [3]:
def ler(caminho: Path) -> pd.DataFrame:
    return pd.read_csv(caminho, sep=';', decimal=',', encoding='utf-8-sig')

ind = ler(PASTA_MODELO / 'fato_indicadores_distrito.csv')
dim_programa = ler(PASTA_MODELO / 'dim_programa.csv')
equip = ler(PASTA_SAIDA_TRAT / 'dim_equipamento.csv')
acesso = ler(PASTA_SAIDA_TRAT / 'fato_acesso_distrito_servico.csv')

arq_ida = PASTA_SAIDA_TRAT / 'extras' / 'fato_indice_ida.csv'
if arq_ida.exists():
    ind = ind.merge(ler(arq_ida)[['cod_distrito', 'moradias_r3r4']], on='cod_distrito', how='left')
else:
    print('Sem saida/extras/fato_indice_ida.csv: risco por distrito estimado só pelo nº de setores R3/R4.')
    ind['moradias_r3r4'] = np.nan

dist = acesso.pivot_table(index='cod_distrito', columns='servico', values='dist_km')
ind = ind.merge(dist.add_prefix('dist_').add_suffix('_km').reset_index(), on='cod_distrito', how='left')

assert len(ind) == 96, f'esperava 96 distritos, vieram {len(ind)}'
id_prog = dict(zip(dim_programa.programa, dim_programa.id_programa))
print(f"{len(ind)} distritos | {int(equip.eh_unidade_app.astype(str).str.lower().eq('true').sum())} unidades do app (UBS/CRAS)")

96 distritos | 635 unidades do app (UBS/CRAS)


## 2. Probabilidades por distrito (o que vem do dado real)

In [4]:
d = ind.set_index('cod_distrito')

# risco: moradias R3/R4 ÷ domicílios, com sobre-representação do CadÚnico
domicilios = d.pop_total / PESSOAS_POR_DOMICILIO
p_risco = (d.moradias_r3r4 / domicilios * SOBRE_REPRESENTACAO_RISCO)
p_risco = p_risco.fillna(pd.Series(np.where(d.setores_risco_r3r4 > 0, 0.08, 0.0), index=d.index)).clip(0, 0.6)
p_risco[d.setores_risco_r3r4.fillna(0) == 0] = 0.0

# 65+: % do distrito no Censo
p_65 = (d.pop_65_mais / d.pop_total * SOBRE_REPRESENTACAO_65).clip(0.05, 0.45)

# pobreza relativa: mais CadÚnico por habitante = renda mais baixa
cad_mil = d.familias_cadunico / d.pop_total * 1000
pobreza = ((cad_mil - cad_mil.median()) / cad_mil.std()).clip(-2, 2)

# já recebe: cobertura real ÷ esperada (lacuna do modelo). Razão 1 = recebe como os parecidos
razao_pbf = (d.cobertura_pbf / d.cobertura_pbf_esperada).replace([np.inf, -np.inf], np.nan).fillna(1).clip(0.5, 1.2)
razao_bpc = (d.taxa_bpc_idoso / d.bpc_idoso_esperado_taxa).replace([np.inf, -np.inf], np.nan).fillna(1).clip(0.5, 1.2)
p_recebe_pbf = (0.80 * razao_pbf).clip(0.35, 0.95)
p_recebe_bpc = (0.75 * razao_bpc).clip(0.30, 0.95)

# documento: distância média ao cartório e ao Poupatempo
dist_doc = d[['dist_Cartório_km', 'dist_Poupatempo_km']].mean(axis=1).fillna(d.dist_cras_km)
p_sem_doc = (P_SEM_DOC_BASE + P_SEM_DOC_POR_KM * dist_doc).clip(0.08, 0.40)

prob = pd.DataFrame({'p_risco': p_risco, 'p_65': p_65, 'pobreza': pobreza, 'p_recebe_pbf': p_recebe_pbf,
                     'p_recebe_bpc': p_recebe_bpc, 'p_sem_doc': p_sem_doc})
display(prob.describe().round(3)) if 'display' in globals() else print(prob.describe().round(3))

       p_risco    p_65  pobreza  p_recebe_pbf  p_recebe_bpc  p_sem_doc
count   96.000  96.000   96.000        96.000        96.000     96.000
mean     0.016   0.169    0.032         0.773         0.721      0.165
std      0.029   0.051    0.836         0.117         0.135      0.034
min      0.000   0.091   -1.528         0.400         0.455      0.123
25%      0.000   0.130   -0.562         0.726         0.606      0.145
50%      0.004   0.162    0.000         0.787         0.723      0.158
75%      0.019   0.211    0.589         0.827         0.854      0.174
max      0.157   0.299    2.000         0.950         0.900      0.378


## 3. Casos (`dim_caso`)

In [5]:
peso = d.familias_cadunico / d.familias_cadunico.sum()
# 1 caso garantido por distrito (nenhum fica em branco no mapa); o resto proporcional ao CadÚnico
cods = np.concatenate([d.index.values, rng.choice(d.index.values, size=N_CASOS - len(d), p=peso.values)])
rng.shuffle(cods)
casos = pd.DataFrame({'id_caso': np.arange(1, N_CASOS + 1), 'cod_distrito': cods})

# unidade: UBS ou CRAS do próprio distrito; se não houver, uma da mesma subprefeitura
equip['eh_unidade_app'] = equip.eh_unidade_app.astype(str).str.lower().eq('true')
unid = equip[equip.eh_unidade_app]
por_dist = unid.groupby('cod_distrito').id_equipamento.apply(list).to_dict()
por_sub = unid.groupby('subprefeitura').id_equipamento.apply(list).to_dict()
def sorteia_unidade(cod):
    lista = por_dist.get(cod) or por_sub.get(d.at[cod, 'subprefeitura']) or unid.id_equipamento.tolist()
    return rng.choice(lista)
casos['id_unidade'] = casos.cod_distrito.map(sorteia_unidade)

# idade: 65+ pelo Censo do distrito, o resto pelas premissas
eh_65 = rng.random(N_CASOS) < casos.cod_distrito.map(p_65).values
outras = rng.choice(FAIXAS_IDADE[:4], size=N_CASOS, p=PROP_IDADE_SEM_65)
casos['faixa_etaria'] = np.where(eh_65, '65+', outras)

# renda: perfil do CadÚnico, deslocado pela pobreza relativa do distrito
def sorteia_renda(z):
    p = PROP_RENDA * np.exp(-0.25 * z * np.arange(len(PROP_RENDA)) * (np.arange(len(PROP_RENDA)) < 5))
    return rng.choice(FAIXAS_RENDA, p=p / p.sum())
casos['faixa_renda'] = casos.cod_distrito.map(pobreza).map(sorteia_renda)

casos['area_risco_r3r4'] = np.where(rng.random(N_CASOS) < casos.cod_distrito.map(p_risco).values, 'Sim', 'Não')
casos['situacao_caso'] = np.where(rng.random(N_CASOS) < PCT_CASOS_ENCERRADOS, 'Encerrado', 'Ativo')
casos['faixa_etaria_ordem'] = casos.faixa_etaria.map({f: i for i, f in enumerate(FAIXAS_IDADE, 1)})
casos['faixa_renda_ordem'] = casos.faixa_renda.map({f: i for i, f in enumerate(FAIXAS_RENDA, 1)})
casos['origem'] = 'SIMULADO'

## 4. Documentos que faltam (`fato_documento_pendente`)

In [6]:
sem_doc = rng.random(N_CASOS) < casos.cod_distrito.map(p_sem_doc).values
linhas = []
docs, pesos_doc = list(PROP_DOCUMENTO), np.array(list(PROP_DOCUMENTO.values()))
for id_caso, idade in casos.loc[sem_doc, ['id_caso', 'faixa_etaria']].itertuples(index=False):
    n = 1 + (rng.random() < 0.35) + (rng.random() < 0.10)      # 1 a 3 documentos
    escolha = rng.choice(docs, size=n, replace=False, p=pesos_doc / pesos_doc.sum())
    for doc in escolha:
        if idade in ('0–5', '6–17') and doc in ('Título de eleitor', 'Carteira de trabalho'):
            continue
        linhas.append({'id_caso': id_caso, 'documento': doc, 'origem': 'SIMULADO'})
fato_doc = pd.DataFrame(linhas).drop_duplicates(['id_caso', 'documento'])
tem_doc_pendente = set(fato_doc.id_caso)

## 5. Plano de Direitos (`fato_item_plano`)
As sugestões seguem as regras de entrada dos 7 programas (as mesmas do simulador do app, em versão resumida); a situação marcada pelo agente segue as probabilidades do passo 2.

In [7]:
ATE = {f: i for i, f in enumerate(FAIXAS_RENDA)}   # 0 = até 218 ... 4 = acima, 5 = não informado
def renda_ate(faixa, limite):                       # não informado entra como possível direito (agente confirma)
    return ATE[faixa] <= ATE[limite] or faixa == 'não informado'

def situacao(p_recebe, id_caso):
    if rng.random() < p_recebe:
        return 'Já recebe'
    if id_caso in tem_doc_pendente and rng.random() < 0.5:
        return 'Falta documento'
    return rng.choice(list(PROP_NAO_RECEBE), p=list(PROP_NAO_RECEBE.values()))

itens = []
for c in casos.itertuples(index=False):
    pr = prob.loc[c.cod_distrito]
    sugest = {}
    if c.id_caso in tem_doc_pendente:
        sugest['Documentação básica'] = 'Falta documento'
    if renda_ate(c.faixa_renda, 'até R$ 810'):
        sugest['CadÚnico'] = situacao(P_RECEBE_FIXO['CadÚnico'], c.id_caso)
        sugest['Tarifa Social de Energia'] = situacao(P_RECEBE_FIXO['Tarifa Social de Energia'], c.id_caso)
    if renda_ate(c.faixa_renda, 'até R$ 218'):
        sugest['Bolsa Família'] = situacao(pr.p_recebe_pbf, c.id_caso)
    if c.faixa_etaria == '65+' and renda_ate(c.faixa_renda, 'até R$ 405'):
        sugest['BPC'] = situacao(pr.p_recebe_bpc, c.id_caso)
    if c.faixa_etaria == '6–17' and renda_ate(c.faixa_renda, 'até R$ 218') and rng.random() < 0.35:  # só ensino médio
        sugest['Pé-de-Meia'] = situacao(P_RECEBE_FIXO['Pé-de-Meia'], c.id_caso)
    if c.faixa_etaria in ('30–64', '65+') and rng.random() < (0.55 if c.faixa_etaria == '65+' else 0.20):  # remédio contínuo
        sugest['Farmácia Popular'] = situacao(P_RECEBE_FIXO['Farmácia Popular'], c.id_caso)
    for prog, sit in sugest.items():
        itens.append({'id_caso': c.id_caso, 'id_programa': id_prog[prog], 'programa': prog, 'situacao': sit})

fato_item = pd.DataFrame(itens)
fato_item['situacao_ordem'] = fato_item.situacao.map(ORDEM_SITUACAO)
fato_item['origem'] = 'SIMULADO'

## 6. Encaminhamentos (`fato_encaminhamento`)

In [8]:
cod_do_caso = casos.set_index('id_caso').cod_distrito
docs_do_caso = fato_doc.groupby('id_caso').documento.apply(list).to_dict()
enc = []
for it in fato_item[fato_item.situacao != 'Já recebe'].itertuples(index=False):
    if rng.random() > P_ENCAMINHA:
        continue
    destino = DESTINO[it.programa]
    if destino is None:
        destino = 'Cartório' if 'Certidão de nascimento' in docs_do_caso.get(it.id_caso, []) else 'Poupatempo'
    km = d.at[cod_do_caso[it.id_caso], f'dist_{DIST_DESTINO[destino]}_km']
    km = 2.0 if pd.isna(km) else km
    p_falta = min(0.5, P_FALTA_BASE + P_FALTA_POR_KM * km)
    if rng.random() < p_falta:
        resultado = 'Faltou'
    else:
        resultado = rng.choice(list(PROP_RESULTADO_SEM_FALTA), p=list(PROP_RESULTADO_SEM_FALTA.values()))
    enc.append({'id_caso': it.id_caso, 'id_programa': it.id_programa, 'tipo_destino': destino, 'resultado': resultado})

fato_enc = pd.DataFrame(enc)
fato_enc.insert(0, 'id_encaminhamento', np.arange(1, len(fato_enc) + 1))
fato_enc['resultado_ordem'] = fato_enc.resultado.map(ORDEM_RESULTADO)
fato_enc['origem'] = 'SIMULADO'

## 6b. Coerência entre plano, encaminhamento e documentos
- Encaminhamento **Concluído** → o item vira **Já recebe**. Na Documentação básica, os documentos saem de `fato_documento_pendente` (e quem não tem mais documento pendente não fica com item travado em "Falta documento").
- Quem **recebe** Bolsa Família, BPC ou Tarifa Social já está no CadÚnico: se o CadÚnico estiver "Ainda não pediu", vira **Precisa atualizar**.


In [ ]:
conc = fato_enc.loc[fato_enc.resultado.eq('Concluído'), ['id_caso', 'id_programa']]
chave = pd.MultiIndex.from_frame(fato_item[['id_caso', 'id_programa']])
resolvido = chave.isin(pd.MultiIndex.from_frame(conc))
fato_item.loc[resolvido, 'situacao'] = 'Já recebe'

docs_ok = set(conc.loc[conc.id_programa.eq(id_prog['Documentação básica']), 'id_caso'])
antes = len(fato_doc)
fato_doc = fato_doc[~fato_doc.id_caso.isin(docs_ok)]
tem_doc_pendente = set(fato_doc.id_caso)
travado = fato_item.situacao.eq('Falta documento') & ~fato_item.id_caso.isin(tem_doc_pendente)
fato_item.loc[travado, 'situacao'] = 'Ainda não pediu'

recebe = set(fato_item.loc[fato_item.programa.isin(['Bolsa Família', 'BPC', 'Tarifa Social de Energia'])
                           & fato_item.situacao.eq('Já recebe'), 'id_caso'])
cad = fato_item.programa.eq('CadÚnico') & fato_item.situacao.eq('Ainda não pediu') & fato_item.id_caso.isin(recebe)
fato_item.loc[cad, 'situacao'] = 'Precisa atualizar'
fato_item['situacao_ordem'] = fato_item.situacao.map(ORDEM_SITUACAO)

print(f'concluídos -> Já recebe: {resolvido.sum()} | documentos resolvidos: {antes - len(fato_doc)} linhas'
      f' | "Falta documento" destravados: {travado.sum()} | CadÚnico -> Precisa atualizar: {cad.sum()}')


## 6c. Lembretes por WhatsApp (premissas)
Não há dado público sobre isso: tudo aqui é **premissa**, para demonstrar as análises de *Efetividade do atendimento* e *Cobertura dos lembretes*.
- Ter WhatsApp (próprio ou de familiar) depende da faixa etária; idosos têm menos e preferem áudio.
- Quem falta mais tende a não ter aceitado lembrete. **Por isso a diferença de faltas com × sem lembrete é efeito da simulação, não um resultado.**


In [ ]:
P_WHATSAPP = {'0–5': (.70, .25), '6–17': (.70, .25), '18–29': (.86, .08), '30–64': (.78, .12), '65+': (.38, .32)}  # (próprio, de familiar)
P_ACEITA = {'WhatsApp próprio': .80, 'WhatsApp de familiar': .68, 'Sem WhatsApp': 0}
FORMATOS = ['Texto', 'Áudio', 'Texto e áudio']
P_FORMATO = {'65+': [.25, .35, .40], 'outros': [.55, .15, .30]}
P_ENVIO = 0.93   # parte dos lembretes não sai (número errado, mensagem não entregue)

def sorteia_whatsapp(fe):
    pp, pf = P_WHATSAPP[fe]; u = rng.random()
    return 'WhatsApp próprio' if u < pp else 'WhatsApp de familiar' if u < pp + pf else 'Sem WhatsApp'
casos['whatsapp'] = casos.faixa_etaria.map(sorteia_whatsapp)

taxa_falta = fato_enc.assign(f=fato_enc.resultado.eq('Faltou')).groupby('id_caso').f.mean()
fr = casos.id_caso.map(taxa_falta).fillna(taxa_falta.mean())
aceita = rng.random(len(casos)) < casos.whatsapp.map(P_ACEITA) * (1 - 0.55 * fr)
casos['aceita_lembrete'] = np.where(casos.whatsapp.eq('Sem WhatsApp'), 'Sem como receber', np.where(aceita, 'Sim', 'Não'))
casos['formato_lembrete'] = [rng.choice(FORMATOS, p=P_FORMATO['65+' if fe == '65+' else 'outros']) if a == 'Sim' else ''
                             for fe, a in zip(casos.faixa_etaria, casos.aceita_lembrete)]

ac = casos.set_index('id_caso').aceita_lembrete; fo = casos.set_index('id_caso').formato_lembrete
enviado = fato_enc.id_caso.map(ac).eq('Sim') & (rng.random(len(fato_enc)) < P_ENVIO)
fato_enc['lembrete_enviado'] = np.where(enviado, 'Sim', 'Não')
fato_enc['formato_lembrete'] = np.where(enviado, fato_enc.id_caso.map(fo), '')

at = casos.situacao_caso.eq('Ativo')
print(pd.crosstab(casos[at].whatsapp, casos[at].aceita_lembrete, margins=True))
print(pd.crosstab(fato_enc.lembrete_enviado, fato_enc.resultado, normalize='index').round(3))


## 7. Alertas cruzados (`fato_alertas`)
Cada regra junta um dado do território (real ou do modelo) com o que o app registrou. Os cortes são relativos (quartis entre os distritos), então não dependem do tamanho da simulação.

In [9]:
ativos = casos[casos.situacao_caso == 'Ativo']
it_a = fato_item[fato_item.id_caso.isin(ativos.id_caso)].merge(ativos[['id_caso', 'cod_distrito']], on='id_caso')
en_a = fato_enc[fato_enc.id_caso.isin(ativos.id_caso)].merge(ativos[['id_caso', 'cod_distrito']], on='id_caso')
dc_a = fato_doc[fato_doc.id_caso.isin(ativos.id_caso)].merge(ativos[['id_caso', 'cod_distrito', 'area_risco_r3r4']], on='id_caso')

por_d = pd.DataFrame(index=d.index)
por_d['casos'] = ativos.groupby('cod_distrito').size()
por_d['nao_pediu'] = it_a[it_a.situacao.eq('Ainda não pediu') & it_a.programa.isin(['BPC', 'Bolsa Família'])].groupby('cod_distrito').size()
por_d['risco_sem_doc'] = dc_a[dc_a.area_risco_r3r4.eq('Sim')].groupby('cod_distrito').id_caso.nunique()
cras = en_a[en_a.tipo_destino.eq('CRAS')]
por_d['falta_cras'] = cras.resultado.eq('Faltou').groupby(cras.cod_distrito).mean()
pbf = it_a[it_a.programa.eq('Bolsa Família')]
por_d['bloqueio_pbf'] = pbf.situacao.isin(['Bloqueado', 'Precisa atualizar']).groupby(pbf.cod_distrito).sum()
por_d['sem_acesso_por_caso'] = it_a[it_a.situacao.ne('Já recebe')].groupby('cod_distrito').size() / por_d.casos
por_d = por_d.fillna(0).join(d[['distrito', 'classe_prioridade', 'dist_cras_km']])
por_d = por_d[por_d.casos >= 10]   # distrito com poucos casos não gera alerta

q = lambda c, p: por_d[c].quantile(p)  # noqa: E731
br = lambda x, k=1: f'{x:,.{k}f}'.replace(',', 'X').replace('.', ',').replace('X', '.')  # noqa: E731
alertas = []
def add(cod, regra, texto, grav, n):
    alertas.append({'cod_distrito': cod, 'regra': regra, 'alerta': texto, 'gravidade': grav, 'n': n})

for cod, r in por_d.iterrows():
    if r.classe_prioridade in ('urgente', 'alta') and r.nao_pediu >= max(3, q('nao_pediu', .75)):
        add(cod, 'Prioridade alta + direito não pedido',
            f'{r.distrito}: {int(r.nao_pediu)} pessoas com BPC ou Bolsa Família sugerido ainda não pediram. Busca ativa.', 'alta', r.nao_pediu)
    if r.risco_sem_doc >= max(5, q('risco_sem_doc', .75)):
        add(cod, 'Risco + documento',
            f'{r.distrito}: {int(r.risco_sem_doc)} casos em área R3/R4 sem documento. Visita prioritária.', 'alta', r.risco_sem_doc)
    if r.dist_cras_km >= d.dist_cras_km.quantile(.75) and r.falta_cras >= max(0.2, q('falta_cras', .75)):
        add(cod, 'Distância + falta',
            f'{r.distrito}: CRAS a {br(r.dist_cras_km)} km e {br(r.falta_cras * 100, 0)}% de faltas. Avaliar CRAS itinerante.', 'média', r.falta_cras * 100)
    if r.bloqueio_pbf >= max(3, q('bloqueio_pbf', .90)):
        add(cod, 'Bloqueio no Bolsa Família',
            f'{r.distrito}: {int(r.bloqueio_pbf)} Bolsa Família bloqueados ou para atualizar. Mutirão de atualização.', 'média', r.bloqueio_pbf)
    if r.classe_prioridade in ('média', 'baixa') and r.sem_acesso_por_caso >= q('sem_acesso_por_caso', .90):
        add(cod, 'Modelo subestimando',
            f'{r.distrito}: prioridade {r.classe_prioridade} no modelo, mas {br(r.sem_acesso_por_caso, 1)} direitos sem acesso por pessoa no app. Revisar.', 'média', r.sem_acesso_por_caso)

fato_alertas = pd.DataFrame(alertas)
fato_alertas = fato_alertas.sort_values(['gravidade', 'n'], ascending=[True, False], key=lambda s: s.map({'alta': 0, 'média': 1}) if s.name == 'gravidade' else s)
fato_alertas['ordem'] = np.arange(1, len(fato_alertas) + 1)
fato_alertas = fato_alertas.drop(columns='n')
fato_alertas['origem'] = 'SIMULADO + TERRITÓRIO'

## 8. Salvar e conferir

In [10]:
def salvar(df, nome):
    df.to_csv(PASTA_APP / f'{nome}.csv', sep=';', decimal=',', index=False, encoding='utf-8-sig')
    print(f'  salvo {nome}.csv ({len(df)} linhas)')

dim_caso = casos[['id_caso', 'cod_distrito', 'id_unidade', 'faixa_etaria', 'faixa_etaria_ordem', 'faixa_renda',
                  'faixa_renda_ordem', 'area_risco_r3r4', 'situacao_caso', 'whatsapp', 'aceita_lembrete',
                  'formato_lembrete', 'origem']]
salvar(dim_caso, 'dim_caso')
salvar(fato_item[['id_caso', 'id_programa', 'situacao', 'situacao_ordem', 'origem']], 'fato_item_plano')
salvar(fato_doc, 'fato_documento_pendente')
salvar(fato_enc, 'fato_encaminhamento')
salvar(fato_alertas[['cod_distrito', 'regra', 'alerta', 'gravidade', 'ordem', 'origem']], 'fato_alertas')

# Conferência: a simulação reproduz o padrão real?
at = dim_caso[dim_caso.situacao_caso == 'Ativo']
print(f'\nPessoas acompanhadas (ativas): {len(at)}')
print(f'Direitos sem acesso: {(fato_item[fato_item.id_caso.isin(at.id_caso)].situacao != "Já recebe").sum()}')
print(f'Pessoas sem documento: {fato_doc[fato_doc.id_caso.isin(at.id_caso)].id_caso.nunique()}')
print(f'Casos em área de risco: {(at.area_risco_r3r4 == "Sim").sum()}')
e = fato_enc[fato_enc.id_caso.isin(at.id_caso)]
print(f'Encaminhamentos concluídos: {(e.resultado == "Concluído").mean():.0%} ({(e.resultado == "Concluído").sum()} de {len(e)})')

# Conferência da calibragem, por programa: o % que já recebe deve subir junto com a cobertura real ÷ esperada
# do modelo (abaixo do esperado = lacuna = menos gente recebendo). Não comparem por classe de prioridade:
# um distrito pode ser urgente por causa do Bolsa Família e não ter lacuna nenhuma de BPC.
chk = fato_item[fato_item.programa.isin(['BPC', 'Bolsa Família'])].merge(casos[['id_caso', 'cod_distrito']], on='id_caso')
razao = {'BPC': razao_bpc, 'Bolsa Família': razao_pbf}
chk['razao'] = [razao[pg].get(cd) for pg, cd in zip(chk.programa, chk.cod_distrito)]
chk['cobertura_no_modelo'] = pd.cut(chk.razao, [0, 0.95, 1.05, 10],
                                    labels=['abaixo do esperado', 'como o esperado', 'acima do esperado'])
tab = chk.groupby(['programa', 'cobertura_no_modelo'], observed=True).situacao.agg(
    pct_ja_recebe=lambda s: round(s.eq('Já recebe').mean() * 100, 1), itens='size').reset_index()
print('\n% que já recebe, pela cobertura real ÷ esperada do modelo no distrito (deve subir de "abaixo" para "acima"):')
print(tab.to_string(index=False))
print(f'\nAlertas gerados: {len(fato_alertas)}')
print(fato_alertas.head(8)[['gravidade', 'alerta']].to_string(index=False))

  salvo dim_caso.csv (5000 linhas)
  salvo fato_item_plano.csv (13404 linhas)
  salvo fato_documento_pendente.csv (1198 linhas)
  salvo fato_encaminhamento.csv (3191 linhas)
  salvo fato_alertas.csv (41 linhas)

Pessoas acompanhadas (ativas): 4482
Direitos sem acesso: 4095
Pessoas sem documento: 758
Casos em área de risco: 119
Encaminhamentos concluídos: 40% (1135 de 2844)

% que já recebe, pela cobertura real ÷ esperada do modelo no distrito (deve subir de "abaixo" para "acima"):
     programa cobertura_no_modelo  pct_ja_recebe  itens
          BPC  abaixo do esperado           55.4    184
          BPC     como o esperado           72.1    122
          BPC   acima do esperado           84.4    141
Bolsa Família  abaixo do esperado           73.5    948
Bolsa Família     como o esperado           79.1   1337
Bolsa Família   acima do esperado           92.2    296

Alertas gerados: 41
gravidade                                                                                          al